In [1]:
#All the imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import norm
from scipy.stats import kurtosis
import os
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.fields import compute_radial_spectrum
from shared.loading import grid_spacing, load, output_times
from shared.paths import BRMS, results_dir, run_dir
from shared.plotting import save_figure, use_style
use_style()
RESULTS = results_dir("quality_checks", "SNR")

In [6]:
# all the runs of the scan, kept for reference
# base_dirs = [
#     f"{base_root}/beta_p_0.0625/beta_e_0.25",
#     f"{base_root}/beta_p_0.0625/beta_e_1",
#     f"{base_root}/beta_p_0.0625/beta_e_4",
#
#     f"{base_root}/beta_p_0.25/beta_e_1",
#     f"{base_root}/beta_p_0.25/beta_e_4",
#     f"{base_root}/beta_p_0.25/beta_e_16",
#
#     f"{base_root}/beta_p_1/beta_e_0.0625",
#     f"{base_root}/beta_p_1/beta_e_0.25",
#     f"{base_root}/beta_p_1/beta_e_1",
#     f"{base_root}/beta_p_1/beta_e_4",
#     f"{base_root}/beta_p_1/beta_e_16",
#
#     f"{base_root}/beta_p_4/beta_e_0.0625",
#     f"{base_root}/beta_p_4/beta_e_0.25",
#     f"{base_root}/beta_p_4/beta_e_1",
#     f"{base_root}/beta_p_4/beta_e_4",
#     f"{base_root}/beta_p_4/beta_e_16",
#
#     f"{base_root}/beta_p_16/beta_e_0.0625",
#     f"{base_root}/beta_p_16/beta_e_0.25",
#     f"{base_root}/beta_p_16/beta_e_16"
# ]

RUNS = [run_dir(beta_p, beta_e=1, brms=BRMS) for beta_p in (0.0625, 0.25, 1, 4, 16)]

# ---- settings ----
EARLY_TIME = 1                       # reference (noise) spectrum, first two figures
LATE_TIME = 200                      # turbulent spectrum
K_DI_MARK = 25                       # k d_i at which the SNR is read off
SNR_THRESHOLD = 3.0
K_RANGE_SHOWN = (5e-2, 50)           # k d_i range of the plots
DENSITY_SPECTRUM_Y_RANGE = (1e-6, 1e-2)

In [4]:
for run in RUNS:
    print(f"Processing: {run}")
    dx, dy = grid_spacing(run)
    run_label = f"{run.parent.name}_{run.name}"
    plot_dir = RESULTS / run_label
    os.makedirs(plot_dir, exist_ok=True)

    # --- Spectra ---
    spectrum_early = compute_radial_spectrum(load("Dn1", EARLY_TIME, run), dx, dy)
    spectrum_late = compute_radial_spectrum(load("Dn1", LATE_TIME, run), dx, dy)

    # =========================
    # Density spectrum plot
    # =========================
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.loglog(spectrum_early.kk, spectrum_early.sp, label=rf'$P_n(t={EARLY_TIME})$')
    ax.loglog(spectrum_late.kk, spectrum_late.sp, label=rf'$P_n(t={LATE_TIME})$')
    ax.set_xlabel(r'$k$')
    ax.set_ylabel(r'$P(k)$')
    ax.set_xlim(*K_RANGE_SHOWN)
    ax.set_ylim(*DENSITY_SPECTRUM_Y_RANGE)
    ax.legend()
    density_plot_path = save_figure(fig, plot_dir, f"density_spectra_t{EARLY_TIME}_t{LATE_TIME}")
    plt.close(fig)

    # =========================
    # SNR plot
    # =========================
    k = spectrum_early.kk
    snr = spectrum_late.sp / spectrum_early.sp
    snr_at_mark = snr[np.argmin(np.abs(k - K_DI_MARK))]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.loglog(k, snr)

    # Threshold line
    ax.axhline(SNR_THRESHOLD, linestyle='--', linewidth=1.2, color='gray', label=f'SNR = {SNR_THRESHOLD:g}')

    # Grey vertical shading where SNR < threshold
    below = snr < SNR_THRESHOLD
    start = None
    for k_index, is_below in enumerate(below):
        if is_below and start is None:
            start = k_index
        elif not is_below and start is not None:
            ax.axvspan(k[start], k[k_index - 1], color='lightgray', alpha=0.5)
            start = None
    if start is not None:
        ax.axvspan(k[start], k[-1], color='lightgray', alpha=0.5)

    # Legend additions
    noise_patch = Patch(facecolor='lightgray', alpha=0.5, label=f'Noise-dominated (SNR < {SNR_THRESHOLD:g})')
    red_dot = Line2D([0], [0], marker='o', color='red', linestyle='None', label=rf'$SNR\ at\ kd_i={K_DI_MARK}$')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles + [noise_patch, red_dot], loc='upper right')

    ax.set_xlabel(r'$k d_i$')
    ax.set_ylabel(r'$\mathrm{SNR}(k d_i)$')
    ax.set_xlim(*K_RANGE_SHOWN)
    snr_plot_path = save_figure(fig, plot_dir, f"snr_t{EARLY_TIME}_t{LATE_TIME}")
    plt.close(fig)

    print("Saved:")
    print(f"  {density_plot_path}")
    print(f"  {snr_plot_path}\n")

Processing: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1
Saved:
  /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1/plots/Density_Spectrum.png
  /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1/plots/SNR.png



In [5]:
for run in RUNS:
    print(f"Processing: {run}")
    dx, dy = grid_spacing(run)

    # --- Spectra ---
    spectrum_early = compute_radial_spectrum(load("Dn1", EARLY_TIME, run), dx, dy)
    spectrum_late = compute_radial_spectrum(load("Dn1", LATE_TIME, run), dx, dy)
    k = spectrum_early.kk
    snr = spectrum_late.sp / spectrum_early.sp

    # ==================================================
    # Find k d_i where the SNR crosses the threshold
    # ==================================================
    crossing_index = None
    for k_index in range(1, len(k)):
        if snr[k_index - 1] >= SNR_THRESHOLD and snr[k_index] < SNR_THRESHOLD:
            crossing_index = k_index
            break

    if crossing_index is None:
        print(f"SNR never drops below {SNR_THRESHOLD:g}.")
        continue

    # ---- Linear interpolation for accurate kdi ----
    k1, k2 = k[crossing_index - 1], k[crossing_index]
    s1, s2 = snr[crossing_index - 1], snr[crossing_index]
    kdi_at_threshold = k1 + (SNR_THRESHOLD - s1) * (k2 - k1) / (s2 - s1)
    print(f"kdi where SNR = {SNR_THRESHOLD:g}  -->  {kdi_at_threshold:.4f}\n")

Processing: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_0.0625/beta_e_1
kdi where SNR = 3  -->  38.5105

Processing: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_0.25/beta_e_1
kdi where SNR = 3  -->  22.8184

Processing: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1
kdi where SNR = 3  -->  14.1252

Processing: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_4/beta_e_1
kdi where SNR = 3  -->  7.3092

Processing: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_16/beta_e_1
kdi where SNR = 3  -->  4.3192



In [4]:
# Same plots with the first output as reference and the noise boundary marked
for run in RUNS:
    print(f"Processing: {run}")
    dx, dy = grid_spacing(run)
    first_time = output_times(run)[0]
    run_label = f"{run.parent.name}_{run.name}"
    plot_dir = RESULTS / run_label
    os.makedirs(plot_dir, exist_ok=True)

    # ------------------------------------------------------------
    # Compute spectra
    # ------------------------------------------------------------
    spectrum_first = compute_radial_spectrum(load("Dn1", first_time, run), dx, dy)
    spectrum_late = compute_radial_spectrum(load("Dn1", LATE_TIME, run), dx, dy)

    # ============================================================
    # Density Spectrum
    # ============================================================
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.loglog(spectrum_first.kk, spectrum_first.sp, label=rf'$P_n(t={first_time})$')
    ax.loglog(spectrum_late.kk, spectrum_late.sp, label=rf'$P_n(t={LATE_TIME})$')
    ax.set_xlabel(r'$k$')
    ax.set_ylabel(r'$P(k)$')
    ax.set_xlim(*K_RANGE_SHOWN)
    ax.set_ylim(*DENSITY_SPECTRUM_Y_RANGE)
    ax.legend()
    density_plot_path = save_figure(fig, plot_dir, f"density_spectra_t{first_time}_t{LATE_TIME}")
    plt.close(fig)

    # ============================================================
    # SNR Plot
    # ============================================================
    k = spectrum_first.kk
    snr = spectrum_late.sp / spectrum_first.sp

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.loglog(k, snr, zorder=3)

    # ------------------------------------------------------------
    # Threshold line
    # ------------------------------------------------------------
    ax.axhline(SNR_THRESHOLD, linestyle='--', linewidth=1.4, color='gray', label=f'SNR = {SNR_THRESHOLD:g}', zorder=2)

    # ------------------------------------------------------------
    # Shade noise region
    # ------------------------------------------------------------
    below = snr < SNR_THRESHOLD
    start = None
    for k_index, is_below in enumerate(below):
        if is_below and start is None:
            start = k_index
        elif not is_below and start is not None:
            ax.axvspan(k[start], k[k_index - 1], color='lightgray', alpha=0.6, zorder=0)
            start = None
    if start is not None:
        ax.axvspan(k[start], k[-1], color='lightgray', alpha=0.6, zorder=0)

    # ------------------------------------------------------------
    # Find LEFT noise boundary (SNR >= threshold -> SNR < threshold)
    # ------------------------------------------------------------
    crossing_index = None
    for k_index in range(1, len(k)):
        if snr[k_index - 1] >= SNR_THRESHOLD and snr[k_index] < SNR_THRESHOLD:
            crossing_index = k_index
            break

    k_border = None
    if crossing_index is not None:
        k1, k2 = k[crossing_index - 1], k[crossing_index]
        s1, s2 = snr[crossing_index - 1], snr[crossing_index]
        k_border = k1 + (SNR_THRESHOLD - s1) * (k2 - k1) / (s2 - s1)
        print(f"SNR={SNR_THRESHOLD:g} boundary at kdi = {k_border:.3f}")

        # Vertical boundary line
        ax.axvline(k_border, color='black', linewidth=1.8, zorder=5, label=f'SNR = {SNR_THRESHOLD:g} boundary')

        # --------------------------------------------------------
        # MARK POSITION ON X-AXIS
        # --------------------------------------------------------
        xticks = list(ax.get_xticks())
        xticks.append(k_border)
        ax.set_xticks(xticks)
        tick_labels = [rf'$k_{{\rm SNR={SNR_THRESHOLD:g}}}={k_border:.1f}$' if np.isclose(tick, k_border) else None
                       for tick in xticks]
        ax.set_xticklabels(tick_labels)
    else:
        print("WARNING: No SNR crossing found.")

    # ------------------------------------------------------------
    # Legend on top
    # ------------------------------------------------------------
    noise_patch = Patch(facecolor='lightgray', alpha=0.6, label=f'Noise-dominated (SNR < {SNR_THRESHOLD:g})')
    handles, labels = ax.get_legend_handles_labels()
    legend = ax.legend(handles=handles + [noise_patch], loc='upper right', frameon=True)
    legend.set_zorder(100)

    # ------------------------------------------------------------
    # Labels
    # ------------------------------------------------------------
    ax.set_xlabel(r'$k d_i$')
    ax.set_ylabel(r'$\mathrm{SNR}(k d_i)$')
    ax.set_xlim(*K_RANGE_SHOWN)
    snr_plot_path = save_figure(fig, plot_dir, f"snr_t{first_time}_t{LATE_TIME}_with_boundary")
    plt.close(fig)

    # ============================================================
    # Done
    # ============================================================
    print("Saved:")
    print(f"  {density_plot_path}")
    print(f"  {snr_plot_path}\n")

Processing: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1
SNR=3 boundary at kdi = 27.734
Saved:
  /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1/plots/Density_Spectrum.png
  /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1/plots/SNR.png



NameError: name 'compute_radial_spectrum' is not defined